General setup

In [69]:
!pip install -q "google-cloud-aiplatform[evaluation]"

In [70]:
from google import genai
from google.genai.types import GenerateContentConfig
import pandas as pd
import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

PROJECT_ID = "qwiklabs-gcp-04-238cff0c99bd"
REGION     = "us-central1"
MODEL      = "gemini-2.5-flash"

vertexai.init(project=PROJECT_ID, location=REGION)

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)
print("Client ready:", PROJECT_ID, REGION, MODEL)

Client ready: qwiklabs-gcp-04-238cff0c99bd us-central1 gemini-2.5-flash


classify the question & Generate the social post - make is usable on X, Insta etc.

In [71]:
%%writefile challenge3.py
"""Challenge 3 functions: question classifier + social post generator."""
from google import genai
from google.genai.types import GenerateContentConfig

PROJECT_ID = "qwiklabs-gcp-04-238cff0c99bd"
REGION     = "us-central1"
MODEL      = "gemini-2.5-flash"

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

CATEGORIES = ["Employment", "General Information", "Emergency Services", "Tax Related"]


def classify_question(question: str) -> str:
    """Classify a user question into exactly one of the four fixed categories."""
    prompt = f"""Classify the user question into EXACTLY ONE of these categories:
{", ".join(CATEGORIES)}

Respond with ONLY the category name, nothing else.

Question: {question}"""
    resp = genai_client.models.generate_content(
        model=MODEL, contents=prompt,
        config=GenerateContentConfig(temperature=0)) #set to 0 so this is as analytical as possible
    result = resp.text.strip()
    for c in CATEGORIES:                 # output to a known label
        if c.lower() in result.lower():
            return c
    return "General Information"         # fallback

def generate_social_post(announcement: str) -> str:
    """Generate a short, friendly social media post for a government announcement."""
    prompt = f"""Write a clear, friendly social media post for the Town of Centenial Colorado
about the following government announcement. Keep it under 280 characters,
include 1-2 relevant hashtags, and use an appropriate tone for the situation.

Announcement: {announcement}"""
    resp = genai_client.models.generate_content(
        model=MODEL, contents=prompt,
        config=GenerateContentConfig(temperature=0.7)) # Set temp so it reads more naturally
    return resp.text.strip()

Overwriting challenge3.py


Unit tests for the classification and social posts

In [72]:
%%writefile test_challenge3.py
"""Unit tests for Challenge 3 — live Gemini calls."""
import pytest
from challenge3 import classify_question, generate_social_post, CATEGORIES


@pytest.mark.parametrize("question,expected", [
    ("How do I apply for a job at city hall?", "Employment"),
    ("Who do I call for a gas leak emergency?", "Emergency Services"),
    ("When are property taxes due?", "Tax Related"),
    ("What time does the town hall open?", "General Information"),
])
def test_classify_returns_correct_category(question, expected):
    assert classify_question(question) == expected



def test_classify_always_returns_valid_label():
    # Even nonsense input must resolve to one of the four categories.
    assert classify_question("asdf qwerty zzz") in CATEGORIES


def test_social_post_is_nonempty_string():
    post = generate_social_post("Annual summer festival this Saturday.")
    assert isinstance(post, str)
    assert len(post) > 0


def test_social_post_reasonable_length():
    post = generate_social_post("Town offices closed Monday for the holiday.")
    assert len(post) <= 320

Overwriting test_challenge3.py


Execute the pytests


In [73]:
!pip install -q pytest
!pytest test_challenge3.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.8.5, typeguard-4.5.2, anyio-4.13.0
collected 7 items                                                              

test_challenge3.py::test_classify_returns_correct_category[How do I apply for a job at city hall?-Employment] PASSED [ 14%]
test_challenge3.py::test_classify_returns_correct_category[Who do I call for a gas leak emergency?-Emergency Services] PASSED [ 28%]
test_challenge3.py::test_classify_returns_correct_category[When are property taxes due?-Tax Related] PASSED [ 42%]
test_challenge3.py::test_classify_returns_correct_category[What time does the town hall open?-General Information] PASSED [ 57%]
test_challenge3.py::test_classify_always_returns_valid_label PASSED      [ 71%]
test_challenge3.py::test_social_post_is_nonempty_string PASSED           [ 85%

Test Announcements

In [74]:
class_eval = pd.DataFrame([
    ("How do I apply for a job with the town?",        "Employment"),
    ("What are the open positions at city hall?",       "Employment"),
    ("Who do I call during a power outage emergency?",  "Emergency Services"),
    ("There's a gas leak on my street, who do I call?", "Emergency Services"),
    ("When are my property taxes due?",                 "Tax Related"),
    ("How do I get a copy of my tax bill?",             "Tax Related"),
    ("What time does the library open?",                "General Information"),
    ("Where is the town hall located?",                 "General Information"),
], columns=["question", "reference"])

def classify_v1(q):   # strict / minimal prompt
    return genai_client.models.generate_content(
        model=MODEL,
        contents=f"Classify into exactly one of {CATEGORIES}. Reply with only the category.\nQuestion: {q}",
        config=GenerateContentConfig(temperature=0)).text.strip()

def classify_v2(q):   # prompt with category definitions
    return genai_client.models.generate_content(
        model=MODEL,
        contents=f"""Categories:
- Employment: jobs, hiring, applications
- General Information: hours, locations, general town info
- Emergency Services: police, fire, outages, hazards
- Tax Related: property tax, tax bills, payments
Reply with ONLY the category name.
Question: {q}""",
        config=GenerateContentConfig(temperature=0)).text.strip()

for name, fn in [("classify_v1 (minimal)", classify_v1), ("classify_v2 (definitions)", classify_v2)]:
    df = class_eval.copy()
    df["response"] = df["question"].apply(fn)
    task = EvalTask(
        dataset=df.rename(columns={"question": "prompt"}),
        metrics=["exact_match"],
    )
    classification_result = task.evaluate()

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 8/8 [00:01<00:00,  7.93it/s]
INFO:vertexai.evaluation._evaluation:All 8 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:1.020140825999988 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 8/8 [00:00<00:00, 10.18it/s]
INFO:vertexai.evaluation._evaluation:All 8 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:0.7924417380017985 seconds


In [75]:
announcements = [
    "Schools closed Friday due to a snowstorm.",
    "Water main repair on Fulton Rd, expect outages 9am-2pm.",
    "City offices closed Monday for the 4th of July holiday.",
    "Annual Rocky Mountain oyster summer festival this Saturday on 16th street mall.",
]

def post_v1(a):   # plain prompt
    return genai_client.models.generate_content(
        model=MODEL, contents=f"Write a city social media post: {a}").text.strip()

def post_v2(a):   # guided prompt: tone + hashtag + length
    return genai_client.models.generate_content(
        model=MODEL,
        contents=f"Write a clear, friendly City of Centennial social post under 280 "
                 f"characters with 1-2 hashtags and an appropriate tone:\n{a}").text.strip()

for name, fn in [("post_v1 (plain)", post_v1), ("post_v2 (guided)", post_v2)]:
    df = pd.DataFrame({"prompt": announcements})
    df["response"] = df["prompt"].apply(fn)
    task = EvalTask(
        dataset=df,
        metrics=[
            MetricPromptTemplateExamples.Pointwise.FLUENCY,
            MetricPromptTemplateExamples.Pointwise.COHERENCE,
            MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
            MetricPromptTemplateExamples.Pointwise.SAFETY,
            MetricPromptTemplateExamples.Pointwise.TEXT_QUALITY
        ],
    )
    result = task.evaluate()

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:36<00:00,  1.84s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:36.89744784799768 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:42<00:00,  2.13s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:42.53527380399828 seconds


Output some pretty results  - Copied from my notes from all the classes/labs ....

In [76]:
import pandas as pd
from IPython.display import display, Markdown

def display_eval_report(eval_result, metrics=None):
    """Display the evaluation results.

    eval_result: a tuple of (title, summary_metrics, metrics_table)
    metrics:      optional list of substrings to filter which metric columns show
    """
    title, summary_metrics, report_df = eval_result

    # summary_metrics is a dict -> turn into a one-row DataFrame
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T

    # optional: keep only columns matching the requested metric names
    if metrics:
        metrics_df = metrics_df.filter(
            [c for c in metrics_df.columns if any(m in c for m in metrics)]
        )
        report_df = report_df.filter(
            [c for c in report_df.columns if any(m in c for m in metrics)]
        )

    display(Markdown(f"## {title}"))
    display(Markdown("### Summary Metrics"))
    display(metrics_df)
    display(Markdown("### Row-by-row Report"))
    display(report_df)

Show the results

In [77]:
display_eval_report((("Social Generation Result", result.summary_metrics, result.metrics_table)))
display_eval_report((("Classification Result", classification_result.summary_metrics, classification_result.metrics_table)))

## Social Generation Result

### Summary Metrics

,row_count,fluency/mean,fluency/std,coherence/mean,coherence/std,instruction_following/mean,instruction_following/std,safety/mean,safety/std,text_quality/mean,text_quality/std
0,4.0,5.0,0.0,5.0,0.0,4.5,1.0,1.0,0.0,3.75,1.5


### Row-by-row Report

,prompt,response,fluency/explanation,fluency/score,coherence/explanation,coherence/score,instruction_following/explanation,instruction_following/score,safety/explanation,safety/score,text_quality/explanation,text_quality/score
0,Schools closed Friday due to a snowstorm.,"Heads up, Centennial! ❄️ Due to the snowstorm,...","The response is free of grammatical errors, de...",5.0,The response demonstrates a seamless logical f...,5.0,The AI response completely fulfills the instru...,5.0,The response is a harmless message about schoo...,1.0,"The response is well-written, coherent, fluent...",3.0
1,"Water main repair on Fulton Rd, expect outages...","Heads up, Centennial! 🛠️ Water main repair on ...","The response is free of grammatical errors, de...",5.0,The response takes the prompt's information an...,5.0,The response fully addresses all aspects of th...,5.0,The response is a helpful and informative mess...,1.0,"The response is exceptionally clear, coherent,...",5.0
2,City offices closed Monday for the 4th of July...,"Heads up, Centennial! City offices will be clo...","The response is free of grammatical errors, de...",5.0,"The response exhibits exceptional coherence, t...",5.0,The AI response fully addresses the user's inp...,5.0,The response is a polite and informative annou...,1.0,"The response is exceptionally clear, coherent,...",5.0
3,Annual Rocky Mountain oyster summer festival t...,Calling all food adventurers! 🍽️ Get ready for...,"The response is free of grammatical errors, de...",5.0,"The response is completely coherent, taking th...",5.0,"The AI correctly identified the event and day,...",3.0,The response is safe as it does not contain an...,1.0,"The response is well-written, coherent, fluent...",2.0


## Classification Result

### Summary Metrics

,row_count,exact_match/mean,exact_match/std
0,8.0,1.0,0.0


### Row-by-row Report

,prompt,reference,response,exact_match/score
0,How do I apply for a job with the town?,Employment,Employment,1.0
1,What are the open positions at city hall?,Employment,Employment,1.0
2,Who do I call during a power outage emergency?,Emergency Services,Emergency Services,1.0
3,"There's a gas leak on my street, who do I call?",Emergency Services,Emergency Services,1.0
4,When are my property taxes due?,Tax Related,Tax Related,1.0
5,How do I get a copy of my tax bill?,Tax Related,Tax Related,1.0
6,What time does the library open?,General Information,General Information,1.0
7,Where is the town hall located?,General Information,General Information,1.0
